# In Vitro CRISPR Model EM-seq: Data Processing and EWAS
**Pershad & Zhao et al., Nature Medicine, 2025**

This notebook documents processing of 5-methylcytosine (5mC) profiling data from primary human CD34+
hematopoietic stem and progenitor cells (HSPCs) carrying engineered TET2 loss-of-function (LoF) or
DNMT3A R882 mutations, generated using CRISPR-Cas9 as described in Kirmani et al. (2025) and Silver et al. (2023).

**Experimental design:**
- TET2 LoF: N=4 donors, TET2 knockout vs. AAVS1 non-targeting control
- DNMT3A R882: N=6 donors, R882H and R882C knock-in vs. AAVS1 control
- Sorted population: CD34+CD38–Lin– (primitive HSC-enriched fraction, Day 7 post-editing)
- Assay: duet evoC (biomodal) with Twist Human Methylome Panel (~4M CpGs, GRCh38)

These in vitro EWAS results define the DMRs used to train the peripheral blood activity scores.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ranksums
import subprocess

## 1. Sample metadata and experimental groups

In [ ]:
# Load sample metadata
metadata = pd.read_table('in_vitro_sample_metadata.tsv')
# Expected columns: sample_id, donor_id, edit_type (TET2_LoF/DNMT3A_R882H/DNMT3A_R882C/Control),
#                   editing_efficiency (%), sequencing_depth

print("Sample counts by edit type:")
print(metadata['edit_type'].value_counts())
print(f"\nTotal samples: {len(metadata)}")
print(f"Unique donors: {metadata['donor_id'].nunique()}")

## 2. Transfer CX_report files from GCS

Raw EM-seq data was processed through the biomodal pipeline (v1.1.1) to produce CX_report files
quantifying the modification state of each CpG site across the genome (GRCh38).

In [ ]:
%%bash
# Copy CX_report files for each CRISPR model batch from GCS

mkdir -p in_vitro_cx_reports/tet2
gsutil -m cp gs://bicklab-main-storage/Data/vantage/11996-JVA/Methylation/CX_report/* \
    in_vitro_cx_reports/tet2/

mkdir -p in_vitro_cx_reports/dnmt3a
gsutil -m cp gs://bicklab-main-storage/Data/vantage/9498-AJ/additional_files/*.CX_report.txt.gz \
    in_vitro_cx_reports/dnmt3a/

echo "TET2 model files: $(ls in_vitro_cx_reports/tet2/*.CX_report.txt.gz | wc -l)"
echo "DNMT3A model files: $(ls in_vitro_cx_reports/dnmt3a/*.CX_report.txt.gz | wc -l)"

## 3. Load and process CX_report data with BSseq

We use the R/Bioconductor `BSseq` package to load, filter, and smooth methylation data.
The code below loads CX_report files, filters to CpGs overlapping the Twist panel target regions,
and applies coverage filters.

In [ ]:
# The primary processing is done in R (shown below as a code block for documentation)
# Run this in an R kernel or via: Rscript in_vitro_bsseq_processing.R

r_code = """
library(BSseq)
library(data.table)
library(GenomicRanges)
library(rtracklayer)
library(BiocParallel)

# Set up parallel processing
bp_param <- BiocParallel::MulticoreParam(workers = min(16, detectCores() - 1))

# Load Twist panel target regions
twist_regions <- rtracklayer::import("covered_targets_Twist_Methylome_hg38_annotated_collapsed.bed")
cat("Twist target regions:", length(twist_regions), "\n")

# Function to read a single CX_report file, filter by target regions and coverage
read_and_filter_cx_report <- function(file_path, target_regions, min_coverage = 5, sample_id = NULL) {
  if (is.null(sample_id)) sample_id <- sub(".CX_report.txt.gz", "", basename(file_path))

  dt <- fread(file_path, header = FALSE, sep = "\t",
              col.names = c("chr", "pos", "strand", "meth_count", "unmeth_count", "context", "trinuc"))

  # Retain CpG-context sites only
  dt <- dt[context == "CG"]

  # Apply coverage filter
  dt[, coverage := meth_count + unmeth_count]
  dt <- dt[coverage >= min_coverage]

  # Filter to Twist panel target regions
  cpg_gr <- GRanges(seqnames = dt$chr, ranges = IRanges(start = dt$pos, width = 1))
  overlaps <- findOverlaps(cpg_gr, target_regions)
  dt <- dt[queryHits(overlaps), ]

  # Create BSseq object
  bs <- BSseq(
    chr  = dt$chr, pos = dt$pos,
    M    = as.matrix(dt$meth_count),
    Cov  = as.matrix(dt$coverage),
    sampleNames = sample_id
  )
  return(bs)
}

# Load all samples and combine
all_files <- c(
  list.files("in_vitro_cx_reports/tet2",   pattern = ".CX_report.txt.gz", full.names = TRUE),
  list.files("in_vitro_cx_reports/dnmt3a", pattern = ".CX_report.txt.gz", full.names = TRUE)
)

bs_list <- lapply(all_files, read_and_filter_cx_report, target_regions = twist_regions, min_coverage = 5)
bs_combined <- do.call(bsseq::combine, bs_list)

saveRDS(bs_combined, "in_vitro_combined_bsseq.rds")
cat("Combined BSseq object:", nrow(bs_combined), "loci x", ncol(bs_combined), "samples\n")
"""
print("R code ready. Run in R or Rscript to process CX_report files.")
print(r_code)

## 4. Identify differentially methylated regions (DMRs)

We segment the genome into 500 bp non-overlapping windows and test for differential methylation
between edited and control cells using the modality framework (Jensen-Shannon Divergence).
DMRs are called at Benjamini-Hochberg adjusted P < 0.05.

For DNMT3A R882, results from R882H and R882C models are intersected, retaining regions
significant in both substitutions.

In [ ]:
# Load EWAS results from modality framework
# (run externally via the modality bioinformatics pipeline)
tet2_dmrs   = pd.read_table('in_vitro_tet2_lof_500bp_dmrs.tsv')   # TET2 LoF vs. Control
r882h_dmrs  = pd.read_table('in_vitro_dnmt3a_r882h_500bp_dmrs.tsv') # R882H vs. Control
r882c_dmrs  = pd.read_table('in_vitro_dnmt3a_r882c_500bp_dmrs.tsv') # R882C vs. Control

# DNMT3A consensus DMRs: significant in both R882H and R882C
sig_r882h = set(r882h_dmrs.loc[r882h_dmrs['padj'] < 0.05, 'region_id'])
sig_r882c = set(r882c_dmrs.loc[r882c_dmrs['padj'] < 0.05, 'region_id'])
dnmt3a_consensus_dmr_ids = sig_r882h & sig_r882c

dnmt3a_dmrs = r882h_dmrs[r882h_dmrs['region_id'].isin(dnmt3a_consensus_dmr_ids)].copy()
# Average 5mC difference and p-value across R882H and R882C for consensus regions
dnmt3a_dmrs['mean_5mc_diff'] = (
    r882h_dmrs.set_index('region_id')['mean_5mc_diff'] +
    r882c_dmrs.set_index('region_id')['mean_5mc_diff']
).loc[dnmt3a_consensus_dmr_ids].values / 2

print(f"TET2 LoF DMRs (padj < 0.05): {(tet2_dmrs['padj'] < 0.05).sum():,}")
print(f"  Hypermethylated: {((tet2_dmrs['padj'] < 0.05) & (tet2_dmrs['mean_5mc_diff'] > 0)).sum():,}")
print(f"  Hypomethylated:  {((tet2_dmrs['padj'] < 0.05) & (tet2_dmrs['mean_5mc_diff'] < 0)).sum():,}")
print(f"\nDNMT3A R882 consensus DMRs (R882H ∩ R882C, padj < 0.05): {len(dnmt3a_dmrs):,}")
print(f"  Hypomethylated:  {(dnmt3a_dmrs['mean_5mc_diff'] < 0).sum():,}")
print(f"  Hypermethylated: {(dnmt3a_dmrs['mean_5mc_diff'] > 0).sum():,}")

In [ ]:
def plot_dmr_volcano(dmrs_df, diff_col='mean_5mc_diff', padj_col='padj',
                     sig_threshold=0.05, diff_threshold=0.1,
                     title='In Vitro EWAS: Differentially Methylated Regions',
                     save_path=None, dpi=300):
    """
    Volcano plot of 500 bp DMR results from in vitro CRISPR EWAS.
    x-axis: mean 5mC difference (mutant – control)
    y-axis: –log10(adjusted P)
    """
    fig, ax = plt.subplots(figsize=(7, 5.5), dpi=dpi)

    log_padj = -np.log10(dmrs_df[padj_col].replace(0, 1e-300))
    diff     = dmrs_df[diff_col]
    thresh   = -np.log10(sig_threshold)

    sig = (log_padj >= thresh) & (diff.abs() >= diff_threshold)
    colors = np.where(sig & (diff > 0), '#d62728',
             np.where(sig & (diff < 0), '#1f77b4', '#b0b0b0'))

    ax.scatter(diff, log_padj, c=colors, s=8, alpha=0.6, edgecolors='none', rasterized=True)
    ax.axhline(thresh, color='gray', linestyle='--', linewidth=0.8)

    n_hyper = int((sig & (diff > 0)).sum())
    n_hypo  = int((sig & (diff < 0)).sum())
    ax.text(0.98, 0.97, f'Hypermeth.: {n_hyper:,}', transform=ax.transAxes,
            ha='right', va='top', color='#d62728', fontsize=10)
    ax.text(0.02, 0.97, f'Hypometh.: {n_hypo:,}', transform=ax.transAxes,
            ha='left', va='top', color='#1f77b4', fontsize=10)

    ax.set_xlabel('Mean 5mC difference (mutant – control)', fontsize=11)
    ax.set_ylabel('–log\u2081\u2080(adjusted P)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    plt.show()
    return fig, ax


# TET2 LoF DMR volcano
plot_dmr_volcano(
    tet2_dmrs,
    title='TET2 Loss-of-Function: In Vitro 500 bp DMRs',
    save_path='figures/tet2_invitro_dmr_volcano.png'
)

# DNMT3A R882 DMR volcano
plot_dmr_volcano(
    dnmt3a_dmrs,
    title='DNMT3A R882 (R882H ∩ R882C): In Vitro 500 bp DMRs',
    save_path='figures/dnmt3a_r882_invitro_dmr_volcano.png'
)

In [ ]:
# Extract CpG site IDs within significant DMRs — input to activity score training
# Restrict to CpGs profiled on Illumina 450K/EPIC arrays for clinical portability

# Load 450K/EPIC probe positions (hg38)
array_probes = pd.read_table('illumina_450k_epic_hg38_probes.tsv')  # chr, pos, probe_id

def get_array_cpgs_in_dmrs(dmrs_df, array_probes_df, padj_col='padj', sig_threshold=0.05):
    """Return probe IDs on standard arrays that fall within significant DMRs."""
    sig_dmrs = dmrs_df[dmrs_df[padj_col] < sig_threshold].copy()
    # Parse region coordinates (expected format: chr:start-end)
    sig_dmrs[['chr', 'coords']] = sig_dmrs['region_id'].str.split(':', expand=True)
    sig_dmrs[['start', 'end']] = sig_dmrs['coords'].str.split('-', expand=True).astype(int)

    # Intersect with array probes
    from functools import reduce
    matched = []
    for _, row in sig_dmrs.iterrows():
        in_dmr = array_probes[
            (array_probes['chr'] == row['chr']) &
            (array_probes['pos'] >= row['start']) &
            (array_probes['pos'] <= row['end'])
        ]
        matched.append(in_dmr)

    if matched:
        return pd.concat(matched).drop_duplicates(subset='probe_id')
    return pd.DataFrame(columns=array_probes.columns)

tet2_array_cpgs   = get_array_cpgs_in_dmrs(tet2_dmrs,   array_probes)
dnmt3a_array_cpgs = get_array_cpgs_in_dmrs(dnmt3a_dmrs, array_probes)

print(f"TET2 DMR CpGs on 450K/EPIC arrays: {len(tet2_array_cpgs):,}")
print(f"DNMT3A DMR CpGs on 450K/EPIC arrays: {len(dnmt3a_array_cpgs):,}")

# Save CpG site lists for R training pipeline
tet2_array_cpgs['probe_id'].to_csv('tet2_dmr_array_cpgs.txt', index=False, header=False)
dnmt3a_array_cpgs['probe_id'].to_csv('dnmt3a_dmr_array_cpgs.txt', index=False, header=False)